Presuponemos aqui que ya disponemos de un modelo entrenado para resolver la tarea. Esot simplemente ejemplifica el flujo de evaluación con los datos de validación, sin intervenir los datos de entrenamiento.

# Libraries

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import subprocess

/home/david/Documents/Master/TFM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Similarity calculation

## Evaluation data

In [ ]:
source = 'validation'  # 'train', 'validation' or 'test'
main_path = f"../data/{source}/english/"

queries_path = main_path + "queries"
corpus_elements_path = main_path + "corpus_elements"
qrels_path = main_path + "qrels.tsv"

queries = pd.read_csv(queries_path,sep="\t")
corpus_elements = pd.read_csv(corpus_elements_path, sep="\t")

In [5]:
queries.head()

,q_id,jobtitle
0,1,nanny
1,2,food technologist
2,3,broadcast engineer
3,4,automation engineer
4,5,veterinarian


### Dictionaries

In [6]:
queries_ids = queries.q_id.to_list()
queries_texts = queries.jobtitle.to_list()
map_queries = dict(zip(queries_ids,queries_texts))

corpus_ids = corpus_elements.c_id.to_list()
corpus_texts = corpus_elements.jobtitle.to_list()
map_corpus = dict(zip(corpus_ids,corpus_texts))

## Model

In [32]:
device = "cpu" # Change to "cuda" if GPU is available
model_path = "all-MiniLM-L6-v2"
print(f"Using device: {device}")
model = SentenceTransformer(model_path, device=device)

Using device: cpu


## Similarities

In [8]:
query_embeddings = model.encode(queries_texts, convert_to_tensor=True, device=device, show_progress_bar=True)
corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True, device=device, show_progress_bar=True)

similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

Batches: 100%|██████████| 82/82 [00:04<00:00, 19.43it/s]


# Evaluation

## Prepare Files

Columnas:

- q_id: Query ID.
- Q0: A constant identifier, usually "Q0".
- doc_id: ID of the retrieved corpus element.
- rank: Position of the corpus element in the ranking.
- score: Relevance score assigned by the model.
- tag: Experiment name

In [ ]:
results = []
model_name = "baseline_model"

for q_idx, q_id in enumerate(queries_ids):
    sorted_indices = np.argsort(-similarities[q_idx])  # Decrease order
    for rank, c_idx in enumerate(sorted_indices):  
        doc_id = corpus_ids[c_idx]
        score = similarities[q_idx, c_idx]
        results.append(f"{str(q_id)} Q0 {str(doc_id)} {rank+1} {score:.4f} {model_name}")

In [29]:
run_file = "./output/evaluation_baseline.trec"

with open(run_file, "w", encoding="utf-8") as f:
    f.write("\n".join(results))

## Evaluation file

In [30]:
command = ["python", "talentclef_evaluate.py", "--qrels", qrels_path, "--run", run_file]

result = subprocess.run(command, capture_output=True, text=True)

if result.stderr:
    print("Error:", result.stderr)
print(result.stdout)

Received parameters:
  qrels: ../data/TaskA/validation/english/qrels.tsv
  run: ./output/evaluation_baseline.trec
Loading qrels...
Loading run...
Running evaluation...

=== Evaluation Results ===
map: 0.2923
mrr: 0.7609
ndcg: 0.4296
precision@5: 0.6762
precision@10: 0.5914
precision@100: 0.0591



In [38]:
import time

today = time.strftime("%Y%m%d_%H%M%S")

with open(f"./output/results_{today}.txt", "w", encoding="utf-8") as f:
    f.write(f"Model path: {model_path}\n\n")
    f.write(f"Model name: {model_name}\n\n")
    f.write("\n".join(result.stdout.splitlines()[-7:]))